# 11 · 地理空間與圖論

用 Python 把「位置」與「關係」化為可計算的資料：先學幾何與地理座標的表達與運算，再學圖論 graph 對網路結構的建模與分析。

## 學習目標
- 能用 shapely 建立並操作向量幾何 vector geometry（點、多邊形、外接矩形），計算面積與空間關係。
- 理解座標參考系統 CRS（Coordinate Reference System）概念，知道面積與距離何時需要座標轉換。
- 能把表格資料組成 GeoDataFrame 概念模型並依屬性上色繪圖。
- 認識 OpenStreetMap（OSM）開放地圖資料與其標籤 tag 系統，理解 osmnx 取得建築與路網的流程，以及離線備援與資料授權。
- 能用空間索引 spatial index 與最近鄰 nearest neighbor 查詢加速大量點的搜尋。
- 能用 networkx 建立圖、計算中心性 centrality 並偵測社群 community。

In [ ]:
# 概念：環境與繪圖字型 setup（本書會畫地圖與圖論結構，先統一設定）
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# 注意：matplotlib 預設字型無中文字，設一個常見中文字型避免圖上中文變方框
matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "SimHei", "Arial Unicode MS"]
matplotlib.rcParams["axes.unicode_minus"] = False   # 讓負號正常顯示，不被字型吃掉

np.random.seed(11)   # 固定亂數種子，讓每次重跑的仿真資料一致，方便對照
print("setup 完成，numpy 版本：", np.__version__)

## 向量幾何基礎 shapely

向量幾何 vector geometry 用座標精確描述空間物件：點 Point、線、多邊形 Polygon。它是地理與空間問題的最小積木。

為什麼先學它：後續所有空間運算（面積、相交、距離）都需要先有「幾何物件」當對象。先掌握如何用座標構造幾何，運算才有施力點。

常見幾何與屬性：

| 物件 | 建立方式 | 常用屬性 |
|---|---|---|
| 點 Point | `Point(x, y)` | `.x`、`.y` |
| 多邊形 Polygon | `Polygon([(x, y), ...])` | `.area`、`.exterior`、`.bounds` |
| 外接矩形 box | `box(minx, miny, maxx, maxy)` | 同 Polygon |

In [ ]:
# 概念：用座標手動建立 Point 與 Polygon，檢視其座標與基本屬性
from shapely.geometry import Point, Polygon, box

# 自造幾個點（座標可想成公尺平面上的位置）
p1 = Point(0, 0)
p2 = Point(3, 4)
print("p2 座標：", p2.x, p2.y)
print("p1 到 p2 的距離：", p1.distance(p2))   # 直線距離，畢氏定理結果應為 5

# 概念：多邊形用「外環座標序列」定義，首尾會自動閉合
square = Polygon([(0, 0), (4, 0), (4, 4), (0, 4)])   # 一個 4x4 的正方形
print("square 面積 area：", square.area)
print("square 外接範圍 bounds：", square.bounds)       # (minx, miny, maxx, maxy)

# 邊界 boundary 是外環線，外環 exterior 是構成外圈的線串
print("外環 exterior 座標：", list(square.exterior.coords))

# 技巧：box 是建立軸對齊矩形的捷徑，等價於手寫四個角
rect = box(1, 1, 3, 2)                               # minx,miny,maxx,maxy
print("box 面積：", rect.area)

## 幾何運算與空間索引

有了幾何物件，就能量測（面積 area、長度）與判斷關係：空間關係 spatial predicate 包含 contains、相交 intersects、相離等。仿射變換 affinity 則可平移、縮放、旋轉幾何。

為什麼需要空間索引 spatial index：當幾何數量很多時，逐一兩兩比對是 O(n²)。STRtree 先用外接框做「粗篩 coarse filter」找出可能相交的候選，再精算，避免無謂比對。

形狀（不執行，僅示意）：

```
tree = STRtree(幾何清單)
候選索引 = tree.query(查詢幾何)   # 只回傳外接框可能相交者
```

In [ ]:
# 概念：空間關係判斷與仿射變換
from shapely.geometry import Point, Polygon, box
from shapely import affinity

plot = box(0, 0, 10, 10)                  # 一塊基地
house = box(2, 2, 5, 6)                   # 基地上的一棟建築

print("基地是否包含建築 contains：", plot.contains(house))
print("兩者是否相交 intersects：", plot.intersects(house))
print("基地是否包含某點：", plot.contains(Point(1, 1)))

# 仿射變換：把建築往右上平移，再放大 1.5 倍（模擬量體調整）
moved = affinity.translate(house, xoff=3, yoff=1)        # 平移
scaled = affinity.scale(house, xfact=1.5, yfact=1.5)     # 以中心縮放
print("平移後 bounds：", moved.bounds)
print("放大後面積：", scaled.area, "（原面積", house.area, "）")

In [ ]:
# 概念：用 STRtree 對數十個小多邊形做粗篩，並與暴力法比對結果
from shapely.geometry import box
from shapely.strtree import STRtree

# 自造 30 個隨機小方塊當作建築足跡
boxes = []
for _ in range(30):
    x, y = np.random.uniform(0, 50, size=2)              # 隨機左下角
    boxes.append(box(x, y, x + 3, y + 3))                # 邊長 3 的小方塊

query_window = box(10, 10, 25, 25)                       # 一個查詢框

tree = STRtree(boxes)                                    # 建空間索引
# 注意：新版 shapely 的 query 回傳「索引」陣列，不是幾何本身
candidate_idx = tree.query(query_window)
# 粗篩只保證外接框可能相交，仍需精算真正相交者
index_hits = [i for i in candidate_idx if boxes[i].intersects(query_window)]

# 暴力法：逐一比對全部 30 個，作為正確答案對照
brute_hits = [i for i, b in enumerate(boxes) if b.intersects(query_window)]

print("STRtree 粗篩候選數：", len(candidate_idx))
print("索引精算相交數：", len(index_hits))
print("暴力法相交數：", len(brute_hits))
print("兩法結果一致：", sorted(index_hits) == sorted(brute_hits))

## 地理座標系統與座標轉換

座標參考系統 CRS（Coordinate Reference System）定義一組座標數字對應地球上哪個位置。同一個點在不同 CRS 下數值完全不同。

兩大類常見 CRS：

| 類型 | 例子 | 單位 | 適合做什麼 |
|---|---|---|---|
| 地理座標 geographic | EPSG:4326（經緯度） | 度 degree | 定位、存取、交換資料 |
| 投影座標 projected | EPSG:3857 等（公尺） | 公尺 metre | 量面積、量距離 |

為什麼重要：面積與距離只有在投影座標下才有正確的公尺意義。直接拿經緯度（度數）算面積是錯的。「何時該轉換」是地理分析的關鍵觀念。

In [ ]:
# 概念：同一點在地理座標與投影座標下的數值差異（pyproj 轉換）
from pyproj import Transformer

# 自造兩個經緯度點（lon 經度, lat 緯度），約在台北一帶
points_lonlat = [(121.5654, 25.0330), (121.5170, 25.0478)]

# 注意：always_xy=True 讓輸入輸出都用 (x=經度, y=緯度) 順序，避免常見的軸顛倒坑
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

for lon, lat in points_lonlat:
    x, y = transformer.transform(lon, lat)               # 度 -> 公尺
    print(f"經緯度 ({lon}, {lat}) -> 投影公尺 ({x:.1f}, {y:.1f})")

# 技巧：兩點距離在投影座標下才有公尺意義；用轉換後座標算才正確
(x0, y0), (x1, y1) = [transformer.transform(lon, lat) for lon, lat in points_lonlat]
dist_m = ((x1 - x0) ** 2 + (y1 - y0) ** 2) ** 0.5
print("兩點投影距離（公尺，含投影變形）：", round(dist_m, 1))

## GeoDataFrame 與地圖繪製 geopandas

GeoDataFrame 是 geopandas 的核心：一張表格 DataFrame 多了一個特殊的 geometry 欄，存放每列的幾何物件，並記錄整體的 CRS。

為什麼好用：它把熟悉的表格操作（篩選、聚合）擴充到幾何，讓屬性資料與位置資料一起處理與視覺化。依某個屬性欄上色畫圖，就是主題地圖 choropleth 的概念。

形狀（不執行，僅示意）：

```
gdf = GeoDataFrame(屬性表, geometry=幾何清單, crs="EPSG:3857")
gdf.plot(column="屬性欄", legend=True)   # 依屬性上色
```

In [ ]:
# 概念：自造區域多邊形組成 GeoDataFrame，依數值上色畫主題地圖
import geopandas as gpd
from shapely.geometry import box

# 自造四個相鄰區域（仿真都市分區）與一欄人口密度數值
regions = [box(0, 0, 2, 2), box(2, 0, 4, 2), box(0, 2, 2, 4), box(2, 2, 4, 4)]
names = ["A", "B", "C", "D"]
density = [120, 340, 80, 500]                            # 仿真人口密度（人/公頃）

# 用幾何清單與屬性建 GeoDataFrame，並標記 CRS 為投影座標（公尺）
gdf = gpd.GeoDataFrame(
    {"name": names, "density": density},
    geometry=regions,
    crs="EPSG:3857",
)
print(gdf)
print("目前 CRS：", gdf.crs)
print("各區面積（由幾何自動算）：", list(gdf.area))

# 依 density 欄上色：數值高的區域顏色較深，即 choropleth
ax = gdf.plot(column="density", cmap="OrRd", legend=True, edgecolor="black")
ax.set_title("各分區人口密度主題地圖 choropleth")
plt.show()

## OpenStreetMap 真實地理資料 osmnx

OpenStreetMap（OSM）是開放的全球地圖資料庫，用標籤 tag（key=value 形式）描述世界，例如 `building=yes`、`highway=primary`、`building:levels=5`、`height=18`。osmnx 能依地名 place 或範圍下載建築足跡 footprint 與路網 street network，回傳成 GeoDataFrame。

工程意識（很重要）：
- 下載需連網，且 OSM 資料受授權 license 規範，使用需標註來源。
- 網路不穩時應有離線 CSV 備援 offline fallback，流程才穩定。
- 標籤常缺漏（並非每棟都有 `building:levels`），要先想好預設值與缺值處理。

下載形狀（不執行，僅示意）：

```
gdf = osmnx.features_from_place("地名", tags={"building": True})
G = osmnx.graph_from_place("地名", network_type="drive")
```

In [ ]:
# 概念：用內建仿真標籤表示意 OSM 流程（不強制連網，含缺值與用途分類）
import pandas as pd

# 注意：真實情境會用 osmnx 下載；這裡用一張內建仿真標籤表當離線備援，
#       讓流程在無網路時仍可示範。每列代表一棟建築的 OSM 標籤。
osm_like = pd.DataFrame([
    {"id": 1, "building": "residential", "building:levels": "5",  "height": None},
    {"id": 2, "building": "commercial",  "building:levels": None, "height": "24"},
    {"id": 3, "building": "yes",         "building:levels": "3",  "height": None},
    {"id": 4, "building": "retail",      "building:levels": None, "height": None},
])

# 一張小型標籤對照表：把 building 標籤映射成用途分類 classification
use_map = {"residential": "住宅", "commercial": "商業", "retail": "商業", "yes": "未知"}
osm_like["use"] = osm_like["building"].map(use_map).fillna("未知")

# 缺值處理：building:levels 缺時，若有 height 就用 height/3 估樓層，否則預設 1 層
def estimate_levels(row):
    if pd.notna(row["building:levels"]):
        return int(row["building:levels"])               # 標籤有值，直接用
    if pd.notna(row["height"]):
        return max(1, round(float(row["height"]) / 3))   # 用高度粗估，每層約 3 公尺
    return 1                                              # 兩者皆缺，保守給 1 層

osm_like["levels_est"] = osm_like.apply(estimate_levels, axis=1)
print(osm_like[["id", "building", "use", "levels_est"]])

## 最近鄰查詢 cKDTree

最近鄰 nearest neighbor 查詢：在一堆點中找「離某點最近的幾個點」。scipy 的 cKDTree 把點建成 KD 樹 KD-tree，讓重複查詢的成本遠低於每次都暴力掃全部。

常見查詢：
- k 最近鄰 k-nearest neighbor：`tree.query(點, k=k)` 回傳最近 k 個的距離與索引。
- 半徑範圍查詢：`tree.query_ball_point(點, r)` 回傳距離 r 內所有點的索引。

為什麼學：空間配對、聚合、群聚分析都建立在「快速找鄰居」之上。

In [ ]:
# 概念：用 cKDTree 查每個點的最近鄰，並與暴力法驗證
from scipy.spatial import cKDTree

# 自造一批二維點（仿真建築點位）
pts = np.random.uniform(0, 100, size=(20, 2))

tree = cKDTree(pts)                                       # 一次建樹，可重複查詢
# 查每個點最近的 2 個（k=2，因為第 1 個一定是它自己，距離 0）
dists, idx = tree.query(pts, k=2)
nearest_idx = idx[:, 1]                                   # 第 2 欄才是真正的最近鄰
nearest_dist = dists[:, 1]

# 暴力法驗證第 0 號點的最近鄰
diff = pts - pts[0]
all_d = np.sqrt((diff ** 2).sum(axis=1))
all_d[0] = np.inf                                         # 排除自己
brute_nearest = int(all_d.argmin())

print("第 0 號點 KD 樹最近鄰索引：", nearest_idx[0])
print("第 0 號點暴力法最近鄰索引：", brute_nearest)
print("一致：", nearest_idx[0] == brute_nearest)

# 半徑查詢：找出第 0 號點 30 單位內的所有鄰居
within = tree.query_ball_point(pts[0], r=30)
print("第 0 號點 30 單位內的點數（含自己）：", len(within))

## 圖論建模 networkx

圖 graph 由節點 node 與邊 edge 組成，用來描述「實體之間的關係」。交通網、社交關係、依賴關係本質都是圖。

幾個關鍵區分：
- 無向圖 undirected：邊沒有方向（朋友關係）；有向圖 directed：邊有方向（追蹤關係）。
- 權重 weight：邊可附數值（距離、成本）。
- 鄰接 adjacency：某節點直接相連的鄰居集合；度 degree 是鄰居數量。

形狀（不執行，僅示意）：

```
G = networkx.Graph()
G.add_edge(a, b, weight=w)   # 從邊清單逐條加入
```

In [ ]:
# 概念：用邊清單建立小型無向加權圖，檢視節點、邊與度
import networkx as nx

# 自造一份邊清單：(節點A, 節點B, 權重) 仿真建築間的鄰近距離
edges = [
    ("A", "B", 5), ("A", "C", 8), ("B", "C", 3),
    ("C", "D", 6), ("D", "E", 2), ("C", "E", 9),
]

G = nx.Graph()                                           # 無向圖
for a, b, w in edges:
    G.add_edge(a, b, weight=w)                           # 加邊時節點自動建立

print("節點 nodes：", list(G.nodes))
print("邊 edges：", list(G.edges(data="weight")))
print("各節點的度 degree：", dict(G.degree()))            # 每個節點的鄰居數

# 畫出圖結構，邊上標權重
pos = nx.spring_layout(G, seed=11)                       # 固定佈局，重跑位置一致
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=800)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "weight"))
plt.title("小型無向加權圖")
plt.show()

## 圖的分析：中心性與社群

建好圖後，常要回答「誰是關鍵節點」與「圖如何自然分群」。中心性 centrality 把節點的重要性量化成數字：

| 中心性 | 衡量什麼 |
|---|---|
| 度中心性 degree | 直接連到多少節點（人脈廣度） |
| 介數中心性 betweenness | 多少最短路徑 shortest path 經過它（橋樑角色） |
| 接近中心性 closeness | 到其他所有節點的平均距離有多近 |

社群偵測 community detection 則找出「內部連結密、彼此連結疏」的節點群，揭示圖的自然分組。

In [ ]:
# 概念：在前一單元的圖上算三種中心性並做一次社群劃分
import networkx as nx

# 重建同一張圖（讓本 cell 自足，不依賴上一格變數）
edges = [("A", "B", 5), ("A", "C", 8), ("B", "C", 3),
         ("C", "D", 6), ("D", "E", 2), ("C", "E", 9)]
G = nx.Graph()
for a, b, w in edges:
    G.add_edge(a, b, weight=w)

deg_c = nx.degree_centrality(G)
bet_c = nx.betweenness_centrality(G)
clo_c = nx.closeness_centrality(G)

print("度中心性 degree：   ", {k: round(v, 2) for k, v in deg_c.items()})
print("介數中心性 betweenness：", {k: round(v, 2) for k, v in bet_c.items()})
print("接近中心性 closeness： ", {k: round(v, 2) for k, v in clo_c.items()})

# 找出最關鍵節點：介數最高者通常是連接不同區塊的橋樑
key_node = max(bet_c, key=bet_c.get)
print("介數最高（最關鍵）節點：", key_node)

# 社群偵測：greedy modularity 把節點分成內部緊密的群
communities = nx.community.greedy_modularity_communities(G)
print("社群劃分結果：", [sorted(c) for c in communities])

## 延伸工具概觀 raster 柵格

前面都在用向量資料 vector（用座標精確描述邊界）。另一大類是柵格資料 raster：把空間切成規則網格，每個像元 pixel 存一個值，像照片或熱度圖。

兩者長處對照：

| 面向 | 向量 vector | 柵格 raster |
|---|---|---|
| 表達 | 精確邊界、面積 | 連續場、密度、覆蓋率 |
| 適合 | 行政界、建築輪廓 | 高程、溫度、密度圖、疊圖分析 |
| 代表工具 | shapely / geopandas | rasterio |

柵格化 rasterize 就是把向量幾何「燒進」網格（rasterio 可做）。何時用：要做密度圖或多層疊圖分析時，網格比逐幾何比對更直接。本單元只建立概念，用簡單網格示意。

In [ ]:
# 概念：用 numpy 網格示意「把建築多邊形轉成覆蓋率網格」（rasterize 概念）
from shapely.geometry import box

# 自造兩棟建築足跡，放在 0~10 的平面上
buildings = [box(1, 1, 4, 3), box(6, 5, 9, 9)]

# 建一個 10x10 的網格，每格 1x1；逐格判斷中心點是否落在任一建築內
grid = np.zeros((10, 10), dtype=int)
for row in range(10):
    for col in range(10):
        center = Point(col + 0.5, row + 0.5)             # 該格中心
        # 注意：這是示意用的逐格掃描；真實大資料應用 rasterio 高效柵格化
        if any(b.contains(center) for b in buildings):
            grid[row, col] = 1                           # 1 表示此格被建築覆蓋

print("被建築覆蓋的格數：", grid.sum(), "/ 100")
plt.imshow(grid, origin="lower", cmap="Greys")           # origin=lower 讓 y 向上
plt.title("建築覆蓋率柵格 raster 示意")
plt.colorbar(label="是否被覆蓋")
plt.show()

## 練習

以下三題由淺入深，皆為複合型但簡短。資料自己用 numpy / list 造，不引用任何外部檔案。

In [ ]:
# TODO 1 ·（簡單）基地占地面積與建蔽率（整合：多邊形 Polygon + 面積 area）
#   情境：手上有一塊街廓基地與其上一棟建築的輪廓，想算出建蔽率
#         coverage（建築占地 footprint 除以基地面積）。
#   要求：
#     1. 用 shapely 的 Polygon，以自己寫死的幾組座標（list）各建一個基地多邊形
#        與一個建築輪廓多邊形。
#     2. 分別取兩者的 .area，相除得到建蔽率。
#   驗收：應該看到 print 出基地面積、建築占地面積，以及一個介於 0 與 1 之間的
#         建蔽率數值。
# TODO: 在這裡完成

In [ ]:
# TODO 2 ·（中間）街廓網格的樓地板面積聚合
#   （整合：點 Point + 包含 contains + 迴圈 / cKDTree 聚合 + 屬性彙總）
#   情境：城市裡有一批建築點位，各自帶有樓層數 floors 與占地面積 footprint，
#         想把它們聚合到所屬的街廓網格 cell，估算每格的樓地板面積
#         GFA（Gross Floor Area，= 占地面積 x 樓層數）。
#   要求：
#     1. 用 numpy 自造數十棟建築的 (x, y) 座標、floors、footprint 三個陣列
#        （仿真數字），並用一組座標切出 2x2 的方形網格 cell
#        （可用 shapely box 或座標範圍判斷）。
#     2. 對每棟建築算出 GFA，並判斷它落在哪一個 cell（用 box.contains(Point)
#        或座標比較）。
#     3. 把同一 cell 內所有建築的 GFA 加總，得到每格的總樓地板面積。
#   驗收：應該看到 print 出 4 個網格各自的總 GFA，且所有網格 GFA 相加等於
#         全部建築 GFA 總和。
# TODO: 在這裡完成

In [ ]:
# TODO 3 ·（稍難）高度政策情境的鄰近影響與關鍵節點
#   （整合：networkx 建圖 + 中心性 centrality + 條件聚合 + 情境前後比較）
#   情境：把彼此距離夠近的建築視為「鄰近關係」連成一張圖 graph，評估一項放寬
#         高度的政策（對特定用途分類的建築套用樓高 height 乘數）後，哪些建築
#         因為位置與鄰里連結最關鍵、且受政策影響最大。
#   要求：
#     1. 用 numpy 自造一批建築的 (x, y) 座標、樓高 height，以及用途分類標籤
#        use（例如 0=住宅、1=商業，仿真即可）。
#     2. 以「兩棟建築距離小於門檻 d 即連一條邊」建出 networkx 無向圖，並計算
#        每個節點的度中心性 degree centrality 與介數中心性 betweenness centrality。
#     3. 套用政策情境：對 use==1（商業）的建築把 height 乘以一個放寬乘數，算出
#        每棟的高度變化量，再以「中心性 x 高度變化量」作為綜合影響分數排序。
#   驗收：應該看到 print 出政策前後的高度對照，以及綜合影響分數最高的前 3 棟
#         建築（同時兼具高中心性與顯著高度增幅）。
# TODO: 在這裡完成

## 小結

- 向量幾何 vector geometry 是空間問題的最小積木：shapely 的 Point、Polygon、box 用座標構造幾何，並提供面積 area、bounds、exterior 等屬性。
- 幾何運算包含空間關係 spatial predicate（contains、intersects）與仿射變換 affinity；幾何很多時用 STRtree 做粗篩避免 O(n²) 暴力比對。
- 座標參考系統 CRS 決定座標數字的意義：地理座標 geographic 是度數、投影座標 projected 是公尺；面積與距離只有在投影座標下才正確，pyproj 負責轉換。
- GeoDataFrame 把表格擴充出 geometry 欄與 CRS，可依屬性上色畫主題地圖 choropleth。
- OpenStreetMap 用標籤 tag 描述世界，osmnx 可取得建築與路網；工程上要注意連網、授權 license、離線備援與標籤缺值處理。
- cKDTree 用 KD 樹 KD-tree 加速最近鄰 nearest neighbor 與半徑查詢，是空間配對與聚合的基礎。
- networkx 用節點與邊建模關係；中心性 centrality（度、介數、接近）量化節點重要性，社群偵測 community detection 揭示自然分群。
- 向量 vector 之外還有柵格 raster：用網格與像元 pixel 表達連續場，rasterio 可做柵格化 rasterize，適合密度圖與疊圖分析。